> # Tutorial: RL Basics

Learning goals:
- Understand the core RL terms: agent, state, action, reward, policy, and episode.
- See the difference between a one-step bandit problem and a sequential decision problem.
- Train a tiny Q-learning agent in a toy environment.


## Outline

1. What reinforcement learning is
2. The key symbols and ideas
3. A tiny multi-armed bandit example
4. Why sequential decisions are harder
5. A simple line-world environment
6. Q-learning step by step
7. Inspecting the learned policy
8. Exercises and pitfalls


## What is Reinforcement Learning?

Reinforcement Learning, or RL, is about learning by trial and error. An **agent** interacts with an **environment**, takes **actions**, receives **rewards**, and gradually improves its behavior.

Unlike supervised learning, the agent is not told the correct answer for each situation. Instead, it has to discover which actions lead to higher long-term reward.

A common mental picture is:

- the agent observes the current situation
- it chooses an action
- the environment responds with a new situation and a reward
- the agent updates what it has learned


### What the symbols mean

RL notation can look abstract at first, so here is the plain-English meaning.

- `s` means a **state**: the current situation the agent is in.
- `a` means an **action**: a choice the agent can make.
- `r` means a **reward**: immediate feedback after an action.
- `pi` means a **policy**: a rule for choosing actions in states.
- `Q(s, a)` means the estimated value of taking action `a` in state `s`.
- an **episode** is one full run, from start until the task ends.

A useful way to think about RL is: the agent is trying to learn a policy that gets high total reward over time, not just a good immediate reward on one step.


In [1]:
from __future__ import annotations

import random
from collections import defaultdict

random.seed(7)


## Example 1: A Tiny Multi-Armed Bandit

A bandit problem is the simplest RL setting. There is no changing state. The agent just picks one of several actions and receives a reward.

Imagine two slot-machine arms:

- arm `A`: usually gives a modest reward
- arm `B`: usually gives a better reward

The challenge is **exploration vs exploitation**:

- exploration means trying actions to learn about them
- exploitation means using what you already believe is best


In [2]:
test_dict = {'A': 1.0, 'B': 1.6, 'C': 1.2, 'D': 2.0}
max(test_dict, key=test_dict.get)

'D'

In [3]:
max(test_dict), min(test_dict)

('D', 'A')

In [4]:
bandit_means = {'A': 1.0, 'B': 1.6}

def pull(arm):
    return round(random.gauss(mu=bandit_means[arm], sigma=0.25), 3)

def epsilon_greedy_bandit(steps=60, epsilon=0.2):
    estimates = {'A': 0.0, 'B': 0.0}
    counts = {'A': 0, 'B': 0}
    history = []

    for step in range(steps):
        if random.random() < epsilon:
            arm = random.choice(['A', 'B'])
        else:
            arm = max(estimates, key=estimates.get)

        reward = pull(arm)
        counts[arm] += 1
        estimates[arm] += (reward - estimates[arm]) / counts[arm]
        history.append((step, arm, reward))

    return counts, estimates, history

counts, estimates, history = epsilon_greedy_bandit()
counts, estimates, history[:8]


({'A': 15, 'B': 45},
 {'A': 1.0922666666666665, 'B': 1.5505111111111107},
 [(0, 'A', 1.212),
  (1, 'A', 1.295),
  (2, 'A', 1.278),
  (3, 'B', 1.706),
  (4, 'B', 1.619),
  (5, 'A', 1.316),
  (6, 'B', 1.375),
  (7, 'B', 1.358)])

In [5]:
history

[(0, 'A', 1.212),
 (1, 'A', 1.295),
 (2, 'A', 1.278),
 (3, 'B', 1.706),
 (4, 'B', 1.619),
 (5, 'A', 1.316),
 (6, 'B', 1.375),
 (7, 'B', 1.358),
 (8, 'A', 1.473),
 (9, 'B', 1.743),
 (10, 'A', 0.711),
 (11, 'B', 1.462),
 (12, 'A', 0.781),
 (13, 'A', 1.226),
 (14, 'B', 1.292),
 (15, 'B', 1.607),
 (16, 'B', 1.81),
 (17, 'B', 1.49),
 (18, 'A', 1.184),
 (19, 'B', 1.704),
 (20, 'B', 1.275),
 (21, 'B', 1.708),
 (22, 'B', 1.767),
 (23, 'A', 1.236),
 (24, 'B', 1.142),
 (25, 'B', 1.374),
 (26, 'B', 1.327),
 (27, 'B', 1.416),
 (28, 'B', 1.866),
 (29, 'B', 1.503),
 (30, 'B', 1.125),
 (31, 'B', 0.97),
 (32, 'B', 1.32),
 (33, 'B', 1.844),
 (34, 'B', 1.376),
 (35, 'B', 1.413),
 (36, 'B', 1.582),
 (37, 'B', 1.349),
 (38, 'B', 1.727),
 (39, 'B', 1.819),
 (40, 'B', 1.733),
 (41, 'B', 1.448),
 (42, 'B', 2.07),
 (43, 'A', 0.582),
 (44, 'A', 1.47),
 (45, 'B', 1.636),
 (46, 'B', 1.265),
 (47, 'B', 1.526),
 (48, 'A', 0.513),
 (49, 'B', 1.736),
 (50, 'B', 1.365),
 (51, 'B', 1.508),
 (52, 'B', 1.468),
 (53, 'B'

The agent begins uncertain, explores both arms, and gradually forms reward estimates. Over time, it should choose arm `B` more often because its average reward is higher.

This is still simpler than full RL because the next situation does not depend on the current action. There is no state transition to worry about.


## Why Sequential Decisions Are Harder

In full RL, actions change the future. A move that looks bad immediately can still be good if it helps the agent reach a better state later.

That is why RL often focuses on **long-term return**, not just the reward from the current step.


## Example 2: A Tiny Line World

We will build a very small environment with 5 positions: `0, 1, 2, 3, 4`.

- the agent starts at position `2`
- action `left` moves one step left
- action `right` moves one step right
- reaching state `4` gives reward `+1` and ends the episode
- reaching state `0` gives reward `-1` and ends the episode
- all other moves give reward `0`

The goal is to learn that moving right is generally better from the middle.


In [6]:
ACTIONS = ['left', 'right']

def step(state, action):
    if action == 'left':
        next_state = max(0, state - 1)
    else:
        next_state = min(4, state + 1)

    if next_state == 4:
        return next_state, 1, True
    if next_state == 0:
        return next_state, -1, True
    return next_state, 0, False

def run_random_episode(start_state=2, max_steps=10):
    state = start_state
    trajectory = []
    for _ in range(max_steps):
        action = random.choice(ACTIONS)
        next_state, reward, done = step(state, action)
        trajectory.append((state, action, reward, next_state))
        state = next_state
        if done:
            break
    return trajectory

run_random_episode()


[(2, 'right', 0, 3), (3, 'right', 1, 4)]

The random agent wanders without learning. To improve, we need a way to estimate which action is valuable in each state.

That is where a Q-table comes in.


## Q-Learning Intuition

A Q-table stores numbers like `Q(state, action)`. Higher values mean the action seems better in that state.

Q-learning updates those values after each step using the idea:

`new estimate = old estimate + learning_rate * (target - old estimate)`

where the target includes the immediate reward plus the best predicted future value from the next state.


In [7]:
def choose_action(q_table, state, epsilon=0.2):
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    values = {action: q_table[(state, action)] for action in ACTIONS}
    return max(values, key=values.get)

def train_q_learning(episodes=250, alpha=0.2, gamma=0.9, epsilon=0.2):
    q_table = defaultdict(float)

    for _ in range(episodes):
        state = 2
        done = False

        while not done:
            action = choose_action(q_table, state, epsilon=epsilon)
            next_state, reward, done = step(state, action)

            best_next = max(q_table[(next_state, next_action)] for next_action in ACTIONS)
            old_value = q_table[(state, action)]
            target = reward + gamma * best_next * (0 if done else 1)
            q_table[(state, action)] = old_value + alpha * (target - old_value)

            state = next_state

    return q_table

q_table = train_q_learning()
{state: {action: round(q_table[(state, action)], 3) for action in ACTIONS} for state in range(5)}


{0: {'left': 0.0, 'right': 0.0},
 1: {'left': -0.945, 'right': 0.808},
 2: {'left': 0.718, 'right': 0.9},
 3: {'left': 0.809, 'right': 1.0},
 4: {'left': 0.0, 'right': 0.0}}

After training, the values for `right` should usually be higher than `left` in the middle states, because moving right is more likely to reach the `+1` terminal state.


In [8]:
learned_policy = {}
for state in [1, 2, 3]:
    values = {action: q_table[(state, action)] for action in ACTIONS}
    learned_policy[state] = max(values, key=values.get)

learned_policy


{1: 'right', 2: 'right', 3: 'right'}

## Test the Learned Policy

Now let the agent act greedily, meaning it always chooses the action with the largest Q-value in each state.


In [9]:
def run_greedy_episode(q_table, start_state=2, max_steps=10):
    state = start_state
    trajectory = []

    for _ in range(max_steps):
        values = {action: q_table[(state, action)] for action in ACTIONS}
        action = max(values, key=values.get)
        next_state, reward, done = step(state, action)
        trajectory.append((state, action, reward, next_state))
        state = next_state
        if done:
            break

    return trajectory

run_greedy_episode(q_table)


[(2, 'right', 0, 3), (3, 'right', 1, 4)]

## Exercises and Pitfalls

Try these:

1. Change `epsilon` from `0.2` to `0.05`. Does the agent explore less?
2. Change the terminal reward at state `4` from `+1` to `+2`. How do the Q-values change?
3. Start the episode from state `1` or `3` and inspect the learned behavior.

Common pitfalls:

- Thinking RL always needs a deep neural network. Many core ideas can be learned with small tables first.
- Focusing only on immediate reward instead of long-term reward.
- Forgetting that exploration is necessary early in training.


In [10]:
# Exercise scaffold: try a different exploration rate.
q_table_low_explore = train_q_learning(episodes=250, epsilon=0.05)
{state: max(ACTIONS, key=lambda action: q_table_low_explore[(state, action)]) for state in [1, 2, 3]}


{1: 'right', 2: 'right', 3: 'right'}